# Notebook 4: 评估与对比

全面评估加权融合方法 vs 多种基线方法。

**基线方法**:
- 加权融合 (分类器概率加权)
- 均匀平均 (各参数等权平均)
- 各单参数方法 (每种参数配置独立估计)
- 默认参数 (未优化的 SolverParams)
- 直接优化基线 (波比跳单参数优化结果, 按 stem 对齐)

**消融研究**: 分布分辨率 (窗口级 vs 全局 vs 均匀)

**P2 修复**: 使用 `burpee_baseline_by_sample` 按 stem 匹配基线。

In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

PROJECT_ROOT = Path(r"D:\data\PPG_HeartRate\Algorithm\outline-PPGtoHR")
ARTIFACTS_DIR = PROJECT_ROOT / "docs" / "research" / "artifacts"
sys.path.insert(0, str(PROJECT_ROOT / "python" / "src"))

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

## 1. 加载融合结果

In [ ]:
# fusion_results 现在是 {stem: fusion_dict} 结构 (P0 修复后的格式)
with open(ARTIFACTS_DIR / "fusion_results.pkl", "rb") as f:
    all_fusion_results = pickle.load(f)

with open(ARTIFACTS_DIR / "simple_params.pkl", "rb") as f:
    params_data = pickle.load(f)

burpee_baseline_by_sample = params_data["burpee_baseline_by_sample"]

print(f"波比文件: {list(all_fusion_results.keys())}")

# 取第一个文件用于详细分析 (如果有多个文件)
sample_stem = list(all_fusion_results.keys())[0]
fusion = all_fusion_results[sample_stem]

t = fusion["t_common"]
ref_bpm = fusion["ref_bpm"]
weighted_bpm = fusion["weighted_hr_bpm"]
uniform_bpm = fusion["uniform_hr_bpm"]
single_hr = fusion["single_hr_bpm"]
motion_mask = fusion["motion_mask"]
rest_mask = ~motion_mask
class_names = fusion["class_names"]
proba = fusion["aligned_proba"]

print(f"当前分析文件: {sample_stem}")
print(f"时间点: {len(t)}, 运动: {motion_mask.sum()}, 静息: {rest_mask.sum()}")
print(f"分类器类别: {class_names}")
print(f"单参数方法: {list(single_hr.keys())}")
print(f"包含默认参数基线: {'是' if 'default_hr_bpm' in fusion else '否'}")

## 2. 基线方法 AAE 对比表

In [ ]:
def compute_aae(pred, ref, mask=None):
    if mask is not None:
        pred, ref = pred[mask], ref[mask]
    return np.mean(np.abs(pred - ref))


# 构建方法字典
methods = {
    "加权融合": weighted_bpm,
    "均匀平均": uniform_bpm,
}
for cn in class_names:
    if cn in single_hr:
        methods[cn] = single_hr[cn]
if "default_hr_bpm" in fusion:
    methods["默认参数"] = fusion["default_hr_bpm"]

# P2: 按 stem 对齐直接优化基线
baseline_rec = burpee_baseline_by_sample.get(sample_stem)
baseline_aae = baseline_rec["min_err_hf"] if baseline_rec else None

# 构建对比表
rows = []
for name, pred in methods.items():
    rows.append({
        "方法": name,
        "AAE_All": compute_aae(pred, ref_bpm),
        "AAE_Rest": compute_aae(pred, ref_bpm, rest_mask),
        "AAE_Motion": compute_aae(pred, ref_bpm, motion_mask),
    })

if baseline_aae is not None:
    rows.append({
        "方法": "直接优化*",
        "AAE_All": baseline_aae,
        "AAE_Rest": float("nan"),
        "AAE_Motion": float("nan"),
    })

df_aae = pd.DataFrame(rows).sort_values("AAE_All")
print(f"文件: {sample_stem}")
display(df_aae.round(2))

## 3. 逐窗口误差分布

In [ ]:
# 构建误差 DataFrame 并绘制总体箱线图
order = df_aae["方法"].tolist()
error_records = []
for name, pred in methods.items():
    errs = np.abs(pred - ref_bpm)
    for i, e in enumerate(errs):
        error_records.append({
            "方法": name,
            "绝对误差": e,
            "阶段": "运动" if motion_mask[i] else "静息",
        })
df_errors = pd.DataFrame(error_records)

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_errors, x="方法", y="绝对误差", order=order, ax=ax)
ax.set_title("各方法逐窗口绝对误差分布")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# 按运动/静息拆分的箱线图
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, phase in zip(axes, ["静息", "运动"]):
    subset = df_errors[df_errors["阶段"] == phase]
    sns.boxplot(data=subset, x="方法", y="绝对误差", order=order, ax=ax)
    ax.set_title(f"{phase}期间误差")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 4. 统计显著性检验

In [ ]:
weighted_errs = np.abs(weighted_bpm - ref_bpm)
uniform_errs = np.abs(uniform_bpm - ref_bpm)

best_single_name = min(
    (cn for cn in class_names if cn in single_hr),
    key=lambda k: compute_aae(single_hr[k], ref_bpm),
)
best_single_errs = np.abs(single_hr[best_single_name] - ref_bpm)

print(f"统计检验 ({sample_stem}):")
print("=" * 60)

_, p1 = stats.wilcoxon(weighted_errs, uniform_errs)
d1 = (np.mean(weighted_errs) - np.mean(uniform_errs)) / np.std(weighted_errs - uniform_errs)
print(f"加权 vs 均匀:  p={p1:.4f}, Cohen's d={d1:.3f}")

_, p2 = stats.wilcoxon(weighted_errs, best_single_errs)
d2 = (np.mean(weighted_errs) - np.mean(best_single_errs)) / np.std(weighted_errs - best_single_errs)
print(f"加权 vs 最佳单参({best_single_name}): p={p2:.4f}, Cohen's d={d2:.3f}")

if "default_hr_bpm" in fusion:
    default_errs = np.abs(fusion["default_hr_bpm"] - ref_bpm)
    _, p3 = stats.wilcoxon(weighted_errs, default_errs)
    d3 = (np.mean(weighted_errs) - np.mean(default_errs)) / np.std(weighted_errs - default_errs)
    print(f"加权 vs 默认参数: p={p3:.4f}, Cohen's d={d3:.3f}")

print("\n注: Cohen's d < 0 表示加权融合误差更小")

## 5. 误差改进与分类器分布

In [ ]:
improvement = uniform_errs - weighted_errs

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                          gridspec_kw={"height_ratios": [2, 1]})

ax = axes[0]
ax.stackplot(range(len(t)),
             *[proba[:, i] for i in range(proba.shape[1])],
             labels=class_names, alpha=0.8)
ax.set_ylabel("概率")
ax.set_title("分类器分布 & 误差改进")
ax.legend(loc="upper right", fontsize=8)

ax = axes[1]
colors = ["green" if v > 0 else "red" for v in improvement]
ax.bar(range(len(t)), improvement, color=colors, alpha=0.6, width=1.0)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("窗口索引")
ax.set_ylabel("误差改进 (BPM)")
ax.set_title("加权融合 vs 均匀平均 (正值=加权更好)")
plt.tight_layout()
plt.show()

## 6. 消融研究: 分布分辨率

In [ ]:
# 全局分布: 用所有窗口的平均概率代替逐窗口概率
global_proba = proba.mean(axis=0, keepdims=True)
global_proba = np.repeat(global_proba, len(t), axis=0)

# 构建单参数心率矩阵
hr_matrix = np.zeros((len(class_names), len(t)))
for i, cn in enumerate(class_names):
    hr_matrix[i] = single_hr.get(cn, ref_bpm)

global_weighted_bpm = np.sum(global_proba * hr_matrix, axis=0)

print("消融: 分布分辨率")
print("-" * 40)
print(f"窗口级分布: AAE={compute_aae(weighted_bpm, ref_bpm):.2f}")
print(f"全局分布:   AAE={compute_aae(global_weighted_bpm, ref_bpm):.2f}")
print(f"均匀平均:   AAE={compute_aae(uniform_bpm, ref_bpm):.2f}")

## 7. 研究总结

In [ ]:
print("=" * 60)
print("研究总结")
print("=" * 60)

# 汇总所有波比文件的结果
print(f"\n波比文件数: {len(all_fusion_results)}")
for stem, f in all_fusion_results.items():
    print(f"\n  [{stem}]")
    print(f"    窗口: {len(f['t_common'])} (运动 {f['motion_mask'].sum()}, 静息 {(~f['motion_mask']).sum()})")
    print(f"    加权融合 AAE: {f['aae_weighted']['all']:.2f} BPM")
    print(f"    均匀平均 AAE: {f['aae_uniform']['all']:.2f} BPM")
    if f.get("baseline_aae") is not None:
        pct = (f["baseline_aae"] - f["aae_weighted"]["all"]) / f["baseline_aae"] * 100
        print(f"    直接优化 AAE: {f['baseline_aae']:.2f} BPM")
        print(f"    相对直接优化: {'+' if pct > 0 else ''}{pct:.1f}%")
    if "default_hr_bpm" in f:
        print(f"    默认参数 AAE: {compute_aae(f['default_hr_bpm'], f['ref_bpm']):.2f} BPM")
    better = f["aae_weighted"]["all"] < f["aae_uniform"]["all"]
    print(f"    分类器是否带来改进: {'是' if better else '否'}")

print(f"\n最佳单参数: {best_single_name} ({compute_aae(single_hr[best_single_name], ref_bpm):.2f} BPM)")